# Cleaning Enrolment Data


Importing the required libraries 

In [13]:
import zipfile
import pandas as pd
import os

<h3><b> Dataset Acquisition and Consolidation </b></h3>


The raw Aadhaar data is stored across multiple compressed fragments to manage file size. We utilize the <b> zipfile </b> and <b> pandas</b> libraries to extract these fragments and merge them into three primary DataFrames:

<b>Biometric Data:</b> ~1.86M records consolidated from 4 source files.

<b>Demographic Data:</b> ~2.07M records consolidated from 5 source files.

<b>Enrolment Data:</b> ~1.00M records consolidated from 3 source files.

In [14]:
zips = {
    'biometric': 'Zipped Datasets/api_data_aadhar_biometric.zip',
    'demographic': 'Zipped Datasets/api_data_aadhar_demographic.zip',
    'enrolment': 'Zipped Datasets/api_data_aadhar_enrolment.zip'
}

# This dictionary will hold the 3 final merged tables
dataframes = {}

for category, zip_name in zips.items():
    print(f"Combining files for: {category}...")
    temp_list = []
    
    try:
        with zipfile.ZipFile(zip_name, 'r') as z:
            # Get all CSV files inside this specific zip
            csv_files = [f for f in z.namelist() if f.endswith('.csv')]
            
            for csv_file in csv_files:
                with z.open(csv_file) as f:
                    # low_memory=False helps with mixed data types in large files
                    temp_df = pd.read_csv(f, low_memory=False)
                    temp_list.append(temp_df)
        
        # Merge the parts (0_500000, 500000_1000000, etc.) into one table
        dataframes[category] = pd.concat(temp_list, ignore_index=True)
        print(f"Successfully created df_{category} with {len(dataframes[category]):,} rows.")
        
    except FileNotFoundError:
        print(f"Error: Could not find {zip_name}. Make sure it's in the same folder!")

df_biometric = dataframes.get('biometric')
df_demographic = dataframes.get('demographic')
df_enrolment = dataframes.get('enrolment')

Combining files for: biometric...
Error: Could not find Zipped Datasets/api_data_aadhar_biometric.zip. Make sure it's in the same folder!
Combining files for: demographic...
Error: Could not find Zipped Datasets/api_data_aadhar_demographic.zip. Make sure it's in the same folder!
Combining files for: enrolment...
Error: Could not find Zipped Datasets/api_data_aadhar_enrolment.zip. Make sure it's in the same folder!


In [15]:
os.makedirs('Raw Datasets', exist_ok=True) 
os.makedirs('Cleaned Datasets', exist_ok=True)

<h3><b>Preliminary Data Inspection and Structural Audit</b></h3>

The primary objectives of this phase include:

<b>Schema Validation:</b> Utilizing .info() to verify column data types and memory usage.

<b>Sampling:</b> Using .head() and .tail() to inspect record consistency across the earliest and latest entries.

<b>Null Distribution Audit:</b> Checking for missing values across all features to determine if imputation is required.

In [16]:
print(df_biometric.head(10))
print(df_biometric.tail(10))

AttributeError: 'NoneType' object has no attribute 'head'

In [ ]:
print(df_demographic.head(10))
print(df_demographic.tail(10))

In [ ]:
print(df_enrolment.head(10))
print(df_enrolment.tail(10))

In [ ]:
df_biometric.info()    

In [ ]:
df_demographic.info()

In [ ]:
df_enrolment.info()

In [ ]:
df_biometric.isnull().sum()

In [ ]:
df_demographic.isnull().sum()

In [ ]:
df_enrolment.isnull().sum()

<b>Record Completeness Check</b>

We perform a secondary audit to identify "invisible" empty records that standard null checks often miss. This targets Empty Strings and Whitespace-only rows to prevent "ghost" categories from appearing in our final district groupings.

In [ ]:
# 1. Check for rows that are completely empty strings ""
empty_strings = (df_demographic == "").all(axis=1).sum()

# 2. Check for rows that are just whitespace " " 
whitespace_rows = df_demographic.apply(lambda x: x.str.strip().eq('') if x.dtype == "object" else False).all(axis=1).sum()

print(f"Total rows that are completely blank: {empty_strings}")
print(f"Total rows that are just whitespace: {whitespace_rows}")

<h3><b>Statistical Profiling & Formatting</b></h3>

We evaluate the scale and distribution of the numeric data across all datasets.

<b>Precision Control:</b> Set global display to two decimal places for better readability.

<b>Descriptive Statistics:</b> Use .describe() to audit the mean, range, and plausibility of population counts for Biometric, Demographic, and Enrolment data.

In [ ]:
pd.options.display.float_format = '{:.2f}'.format
df_biometric.describe()

In [ ]:
pd.options.display.float_format = '{:.2f}'.format
df_demographic.describe()

In [ ]:
pd.options.display.float_format = '{:.2f}'.format
df_enrolment.describe()

<h3><b>Temporal Standardization</b></h3>

String-based date columns are converted into official Python datetime objects to enable chronological analysis.

<b>Unified Format:</b> Transformed all date columns to datetime64[ns] across every dataset.

<b>Accuracy:</b> Used dayfirst=True to ensure correct parsing of the source data structure.

In [ ]:
df_biometric['date'] = pd.to_datetime(df_biometric['date'], dayfirst=True)
df_demographic['date'] = pd.to_datetime(df_demographic['date'], dayfirst=True)
df_enrolment['date'] = pd.to_datetime(df_enrolment['date'], dayfirst=True)

print("Dates converted successfully!")

Verifying the changes made to the dates


In [ ]:
df_biometric.info()    

In [ ]:
df_demographic.info()

In [ ]:
df_enrolment.info()

<h3><b>State-Level Data Audit</b></h3>

We perform an initial aggregation to identify inconsistencies in geographical naming conventions.

<b>Summation:</b> Calculated total to evaluate the scale of data per state.

<b>Error Detection:</b> Grouped the data to pinpoint "ghost" states and spelling duplicates (e.g., "West Bangal" vs "West Bengal").

<b>Targeting:</b> This summary provides the roadmap for the surgical mapping needed to unify the dataset.

In [ ]:
df_enrolment['total_enrol'] = df_enrolment['age_0_5'] + df_enrolment['age_5_17'] + df_enrolment['age_18_greater']
state_enrolment = df_enrolment.groupby('state')['total_enrol'].sum().sort_values(ascending=False)    
print(state_enrolment)

<h3><b>State Name Standardization</b></h3>

We utilize a mapping dictionary to resolve spelling variations and consolidate historical administrative divisions.

<b>Cleaning:</b> Stripped leading/trailing whitespace from state names to ensure exact matches.

<b>Normalization:</b>Unified multiple variants for states like West Bengal, Odisha, and Jammu and Kashmir into single standard entries.

<b>UT Consolidation:</b> Merged fragments of Dadra and Nagar Haveli and Daman and Diu to reflect current Union Territory structures.

<b>Data Filtering:</b> Removed invalid records (e.g., "100000") to maintain dataset integrity.

In [ ]:
df_enrolment['state'] = df_enrolment['state'].str.strip()

cleanup_map = {
    # West Bengal
    'West  Bengal': 'West Bengal', 'West Bangal': 'West Bengal', 
    'West bengal': 'West Bengal', 'Westbengal': 'West Bengal', 
    'WEST BENGAL': 'West Bengal', 'WESTBENGAL': 'West Bengal',
    
    # Andhra Pradesh
    'andhra pradesh': 'Andhra Pradesh',
    
    # Odisha
    'orissa': 'Odisha', 'ODISHA': 'Odisha', 'Orissa': 'Odisha',
    
    # Jammu and Kashmir
    'Jammu And Kashmir': 'Jammu and Kashmir', 'Jammu & Kashmir': 'Jammu and Kashmir',
    
    # Puducherry
    'Pondicherry': 'Puducherry',
    
    # The Big Union Territory Merge
    'Dadra and Nagar Haveli': 'Dadra and Nagar Haveli and Daman and Diu',
    'Daman and Diu': 'Dadra and Nagar Haveli and Daman and Diu',
    'Dadra & Nagar Haveli': 'Dadra and Nagar Haveli and Daman and Diu',
    'Daman & Diu': 'Dadra and Nagar Haveli and Daman and Diu',
    'The Dadra And Nagar Haveli And Daman And Diu': 'Dadra and Nagar Haveli and Daman and Diu',
    
    # Andaman and Nicobar
    'Andaman & Nicobar Islands': 'Andaman and Nicobar Islands'
}

# Apply the mapping
df_enrolment['state'] = df_enrolment['state'].replace(cleanup_map)

# Remove the "100000" garbage value
df_enrolment = df_enrolment[df_enrolment['state'] != '100000']

# Final check
print(df_enrolment['state'].value_counts())

<h3><b>District-Pincode Audit</b></h3>

We analyze the relationship between districts and PIN codes to uncover spelling inconsistencies.

<b>Mapping:</b> Extracted unique pincode arrays for every district entry.

<b>Detection:</b> Sorted the results to identify cases where identical PIN codes were assigned to multiple name variations (e.g., "Chittoor" vs "chittoor").

In [ ]:
district_check = df_enrolment.groupby('district')['pincode'].unique()

district_check.sort_index().tail(20)

In [ ]:
district_check.sort_index().head(20)

<h3><b>Automated Typo Detection via Fuzzy Matching</b></h3>

We implement a string similarity algorithm to identify misspelled district names that passed through initial filters.

<b>Fuzzy Search:</b> Utilized the difflib library to perform fuzzy matching within each state.

<b>Similarity Threshold:</b> Set a 0.8 cutoff to flag names that are highly similar but not identical (e.g., "Gurgaon" vs "Gurugram").

<b>Targeting:</b> The resulting list identifies "ghost" duplicates that require manual mapping to ensure data integrity during aggregation.

In [ ]:
from difflib import get_close_matches

def find_district_typos(df):
    for state in df['state'].unique():
        districts = df[df['state'] == state]['district'].unique()
        for d in districts:
            # Find names that are very similar but not identical
            matches = get_close_matches(d, districts, n=2, cutoff=0.8)
            if len(matches) > 1:
                print(f"In {state}: Potential duplicates {matches}")

find_district_typos(df_enrolment)

<h3><b>Surgical District Standardization</b></h3>

We implement a multi-stage cleanup to resolve naming anomalies and historical name changes across the enrolment dataset.

<b>Normalization:</b> Applied Title Case and stripped whitespace for initial uniformity.

<b>Mapping:</b> Resolved 40+ specific errors, including updating "Allahabad" to "Prayagraj" and consolidating "Bangalore Rural" into "Bengaluru Rural".

<b>Precision:</b> Applied fixes state-specifically to prevent cross-state corruption (e.g., distinguishing between Aurangabad in Bihar vs. Maharashtra).

<b>Refinement:</b> Removed non-alphanumeric noise like asterisks (*) and dots to ensure clean data grouping.

In [ ]:
import pandas as pd

df_enrolment['district'] = df_enrolment['district'].str.strip().str.title()


master_district_corrections = {
    'Karnataka': {
        'Bangalore Rural': 'Bengaluru Rural', 'Ramanagar': 'Ramanagara',
        'Bagalkot *': 'Bagalkot', 'Chamrajanagar': 'Chamarajanagar', 
        'Chamrajnagar': 'Chamarajanagar', 'Chickmagalur': 'Chikkamagaluru',
        'Chikmagalur': 'Chikkamagaluru', 'Davanagere': 'Davangere',
        'Gadag *': 'Gadag', 'Hasan': 'Hassan', 'Haveri *': 'Haveri',
        'Shimoga': 'Shivamogga', 'Tumkur': 'Tumakuru', 'Udupi *': 'Udupi'
    },
    'Uttar Pradesh': {
        'Mahrajganj': 'Maharajganj', 'Bulandshahar': 'Bulandshahr',
        'Bagpat': 'Baghpat', 'Barabanki': 'Bara Banki', 'Shrawasti': 'Shravasti',
        'Siddharthnagar': 'Siddharth Nagar', 'Kushinagar *': 'Kushinagar',
        'Kushi Nagar': 'Kushinagar', 'Raebareli': 'Rae Bareli',
        'Sant Ravidas Nagar Bhadohi': 'Sant Ravidas Nagar'
    },
    'West Bengal': {
        'Coochbehar': 'Cooch Behar', 'Darjiling': 'Darjeeling',
        '24 Paraganas North': 'North 24 Parganas', '24 Paraganas South': 'South 24 Parganas',
        'North Twenty Four Parganas': 'North 24 Parganas', 'South Twenty Four Parganas': 'South 24 Parganas',
        'Barddhaman': 'Bardhaman', 'East Midnapur': 'East Midnapore',
        'Maldah': 'Malda', 'Puruliya': 'Purulia', 'Hawrah': 'Howrah',
        'Hooghiy': 'Hooghly', 'South 24 Pargana': 'South 24 Parganas'
    },
    'Maharashtra': {
        'Ahmadnagar': 'Ahmednagar', 'Ahmed Nagar': 'Ahmednagar',
        'Mumbai( Sub Urban )': 'Mumbai Suburban', 'Buldana': 'Buldhana',
        'Chatrapati Sambhaji Nagar': 'Chhatrapati Sambhajinagar',
        'Gondiya': 'Gondia', 'Gondiya *': 'Gondia', 'Hingoli *': 'Hingoli'
    },
    'Bihar': {
        'Purba Champaran': 'Purbi Champaran', 'Purnea': 'Purnia',
        'Sheikpura': 'Sheikhpura', 'Samstipur': 'Samastipur',
        'Aurangabad(Bh)': 'Aurangabad', 'Aurangabad(Bh)': 'Aurangabad'
    },
    'Odisha': {
        'Anugul': 'Angul', 'Anugal': 'Angul', 'Nabarangapur': 'Nabarangpur',
        'Khorda': 'Khordha', 'Baleshwar': 'Baleswar', 'Baudh': 'Boudh',
        'Jagatsinghapur': 'Jagatsinghpur', 'Jajapur': 'Jajpur', 'Sundergarh': 'Sundargarh'
    },
    'Telangana': {
        'Jangoan': 'Jangaon', 'Medchal-Malkajgiri': 'Medchal-Malkajgiri',
        'Medchal−Malkajgiri': 'Medchal-Malkajgiri', 'Medchal?Malkajgiri': 'Medchal-Malkajgiri',
        'Medchal Malkajgiri': 'Medchal-Malkajgiri'
    },
    'Jammu And Kashmir': {
        'Shupiyan': 'Shopian', 'Badgam': 'Budgam', 'Bandipore': 'Bandipur',
        'Rajauri': 'Rajouri'
    }
}

for state, mapping in master_district_corrections.items():
    for wrong, right in mapping.items():
        # Only fix if BOTH state and district match to avoid cross-state accidents
        mask = (df_enrolment['state'].str.title() == state) & (df_enrolment['district'] == wrong)
        df_enrolment.loc[mask, 'district'] = right

df_enrolment['district'] = df_enrolment['district'].str.replace(r'\s*[*]\s*', '', regex=True).str.strip()

print("District cleanup finished! Legitimate directional districts were preserved.")

<h3><b>Advanced Anomaly Detection</b></h3>

We utilize fuzzy string matching to identify subtle spelling variations and duplicate records.

<b>Fuzzy Logic:</b> Applied difflib to compare district names with an 80% similarity threshold.

<b>Contextual Audit:</b> Scoped comparisons within each state to accurately flag typos like "Gurugram" vs. "Gurgaon" while avoiding cross-border errors.

In [ ]:
from difflib import get_close_matches

def find_district_typos(df):
    for state in df['state'].unique():
        districts = df[df['state'] == state]['district'].unique()
        for d in districts:
            # Find names that are very similar but not identical
            matches = get_close_matches(d, districts, n=2, cutoff=0.8)
            if len(matches) > 1:
                print(f"In {state}: Potential duplicates {matches}")

find_district_typos(df_enrolment)

Mapping again based on previous output

In [ ]:
import pandas as pd

df_enrolment = df_enrolment.copy()


final_comprehensive_map = {
    'Haryana': {'Yamuna Nagar': 'Yamunanagar'},
    'Rajasthan': {
        'Jalor': 'Jalore', 'Dholpur': 'Dhaulpur', 
        'Chittaurgarh': 'Chittorgarh', 'Jhunjhunun': 'Jhunjhunu'
    },
    'Punjab': {
        'S.A.S Nagar(Mohali)': 'Sas Nagar (Mohali)', 
        'Ferozepur': 'Firozpur'
    },
    'Madhya Pradesh': {
        'Ashok Nagar': 'Ashoknagar', 'Narsimhapur': 'Narsinghpur'
    },
    'Assam': {'Sibsagar': 'Sivasagar'},
    'Uttarakhand': {'Hardwar': 'Haridwar'},
    'Gujarat': {
        'Banas Kantha': 'Banaskantha', 'Sabar Kantha': 'Sabarkantha',
        'Panch Mahals': 'Panchmahals', 'Surendra Nagar': 'Surendranagar',
        'Ahmadabad': 'Ahmedabad'
    },
    'Andhra Pradesh': {
        'Visakhapatanam': 'Visakhapatnam', 'Mahabub Nagar': 'Mahabubnagar',
        'Anantapur': 'Ananthapur', 'Ananthapuramu': 'Ananthapur',
        'Karim Nagar': 'Karimnagar', 'K.V.Rangareddy': 'K.V. Rangareddy'
    },
    'Tamil Nadu': {
        'Kancheepuram': 'Kanchipuram', 'Thiruvallur': 'Tiruvallur',
        'Kanniyakumari': 'Kanyakumari', 'Thiruvarur': 'Tiruvarur',
        'Tirupathur': 'Tirupattur', 'Viluppuram': 'Villupuram'
    },
    'Chhattisgarh': {
        'Gaurella Pendra Marwahi': 'Gaurela-Pendra-Marwahi',
        'Janjgir-Champa': 'Janjgir Champa', 'Janjgir - Champa': 'Janjgir Champa',
        'Mohla-Manpur-Ambagarh Chouki': 'Mohalla-Manpur-Ambagarh Chowki'
    },
    'Jharkhand': {
        'Pakaur': 'Pakur', 'Hazaribag': 'Hazaribagh', 
        'East Singhbum': 'East Singhbhum', 'Palamau': 'Palamu',
        'Sahebganj': 'Sahibganj', 'Kodarma': 'Koderma'
    },
    'Telangana': {
        'K.V. Rangareddy': 'Rangareddy', 'Ranga Reddy': 'Rangareddy',
        'Warangal (Urban)': 'Warangal Urban'
    },
    'Mizoram': {'Mammit': 'Mamit'},
    'Kerala': {'Kasargod': 'Kasaragod'},
    'Dadra And Nagar Haveli And Daman And Diu': {
        'Dadra & Nagar Haveli': 'Dadra And Nagar Haveli'
    },
    'Himachal Pradesh': {
        'Lahul And Spiti': 'Lahaul And Spiti', 'Lahul & Spiti': 'Lahaul And Spiti'
    },
    'Andaman And Nicobar Islands': {'Nicobars': 'Nicobar'}
}

for state, mapping in final_comprehensive_map.items():
    for wrong, right in mapping.items():
        mask = (df_enrolment['state'] == state) & (df_enrolment['district'] == wrong)
        df_enrolment.loc[mask, 'district'] = right

df_enrolment['district'] = df_enrolment['district'].str.replace(r'\s*[*]\s*', '', regex=True).str.strip()

print("Massive cleanup complete! All manually identified typos are now standardized.")

Checking for potential duplicates again

In [ ]:
from difflib import get_close_matches

def find_district_typos(df):
    for state in df['state'].unique():
        districts = df[df['state'] == state]['district'].unique()
        for d in districts:
            # Find names that are very similar but not identical
            matches = get_close_matches(d, districts, n=2, cutoff=0.8)
            if len(matches) > 1:
                print(f"In {state}: Potential duplicates {matches}")

find_district_typos(df_enrolment)

Mapping based on previous output

In [ ]:
import pandas as pd

df_enrolment = df_enrolment.copy()

final_fixes = {
    'Andhra Pradesh': {
        'Mahbubnagar': 'Mahabubnagar',
        'Mahabub Nagar': 'Mahabubnagar'
    },
    'Dadra And Nagar Haveli And Daman And Diu': {
        'Dadra & Nagar Haveli': 'Dadra And Nagar Haveli',
        'Dadra And Nagar Haveli': 'Dadra And Nagar Haveli' # Ensuring consistency
    },
    'Andaman And Nicobar Islands': {
        'Nicobars': 'Nicobar'
    }
}

for state, mapping in final_fixes.items():
    for wrong, right in mapping.items():
        mask = (df_enrolment['state'] == state) & (df_enrolment['district'] == wrong)
        df_enrolment.loc[mask, 'district'] = right

df_enrolment['district'] = df_enrolment['district'].str.replace('&', 'And', regex=False)
df_enrolment['district'] = df_enrolment['district'].str.strip()

print("Final cleanup executed. Mahabubnagar, Nicobar, and Dadra variations are now unified!")

Verifying the presence of duplicates again

In [ ]:
from difflib import get_close_matches

def find_district_typos(df):
    for state in df['state'].unique():
        districts = df[df['state'] == state]['district'].unique()
        for d in districts:
            # Find names that are very similar but not identical
            matches = get_close_matches(d, districts, n=2, cutoff=0.8)
            if len(matches) > 1:
                print(f"In {state}: Potential duplicates {matches}")

find_district_typos(df_enrolment)

Finishing up with remaining mapping

In [ ]:
df_enrolment['state'] = df_enrolment['state'].str.strip().str.title()
df_enrolment['district'] = df_enrolment['district'].str.strip().str.title()

mask_an = df_enrolment['state'].str.contains('Andaman', na=False)
df_enrolment.loc[mask_an & (df_enrolment['district'] == 'Nicobars'), 'district'] = 'Nicobar'

mask_dadra = df_enrolment['state'].str.contains('Dadra', na=False)
df_enrolment.loc[mask_dadra & (df_enrolment['district'].str.contains('&')), 'district'] = 'Dadra And Nagar Haveli'

print("Checking Andaman Districts:")
print(df_enrolment[df_enrolment['state'].str.contains('Andaman')]['district'].unique())

Verifying once again

In [ ]:
from difflib import get_close_matches

def find_district_typos(df):
    for state in df['state'].unique():
        districts = df[df['state'] == state]['district'].unique()
        for d in districts:
            # Find names that are very similar but not identical
            matches = get_close_matches(d, districts, n=2, cutoff=0.8)
            if len(matches) > 1:
                print(f"In {state}: Potential duplicates {matches}")

find_district_typos(df_enrolment)

In [ ]:
district_check = df_enrolment.groupby('district')['pincode'].unique()

district_check.sort_index().tail(20)

In [ ]:
district_check.sort_index().head(20)

Cross-Verifying

In [ ]:
df_enrolment['Total_Enrolment'] = df_enrolment['age_0_5'] + df_enrolment['age_5_17'] + df_enrolment['age_18_greater']

state_eda = df_enrolment.groupby('state')['Total_Enrolment'].sum().sort_values(ascending=False).reset_index()

pd.set_option('display.max_rows', None) 
print(state_eda)

In [ ]:
# Group by district and sum the age counts
district_summary = df_enrolment.groupby('district')[['age_0_5', 'age_5_17', 'age_18_greater']].sum()

# Display top 20 districts by total enrollment (calculating total on the fly for sorting)
print(district_summary.assign(Total=district_summary.sum(axis=1)).sort_values('Total', ascending=False).head(20))

In [ ]:
# Drop the redundant total_enrol column
df_enrolment.drop(columns=['total_enrol'], inplace=True)

print("Column 'total_enrol' has been removed.")

In [ ]:
df_enrolment.head(20)

<h3><b>Data Export</b></h3>

The finalized datasets are saved to create a high-integrity baseline for the final merge.

<b>Persistence:</b> Exported the standardized DataFrames to CSV format.

<b>Verification:</b> These clean files serve as the "source of truth" for all subsequent analytical dashboards and master reports.

In [ ]:
df_enrolment.to_csv('Cleaned Datasets/cleaned_enrollment.csv', index=False)

In [ ]:
# Save Demographic data
df_demographic.to_csv('Raw Datasets/raw_demographic.csv', index=False)

# Save Biometric data
df_biometric.to_csv('Raw Datasets/raw_biometric.csv', index=False)

print("Both files saved: 'raw_demographic.csv' and 'raw_biometric.csv'")

<h3><b>Post-Save Quality Audit</b></h3>

We perform a comprehensive re-aggregation of the saved data to verify the effectiveness of the initial cleanup.

<b>Re-Validation:</b> Grouped the dataset by state and district to generate a complete summary of all age groups.

<b>Refinement Strategy:</b> This audit serves as the final checklist for the second phase of surgical name consolidation.

In [ ]:
enrolment_summary = df_enrolment.groupby(['state', 'district'])[['age_0_5', 'age_5_17', 'age_18_greater']].sum().sum(axis=1)

print("--- ENROLLMENT DISTRICT AUDIT ---")
print(enrolment_summary.sort_values(ascending=False).to_string())

In [ ]:
# 1. Surgical mapping for the enrollment survivors
enrollment_final_map = {
    'Bihar': {
        'Kaimur (Bhabua)': 'Kaimur'
    },
    'Karnataka': {
        'Bijapur(Kar)': 'Vijayapura'
    },
    'Andhra Pradesh': {
        'K.V. Rangareddy': 'Rangareddy',
        'Rangareddi': 'Rangareddy'
    }
}

# 2. Apply to df_enrollment
for state, mapping in enrollment_final_map.items():
    for wrong, right in mapping.items():
        mask = (df_enrolment['state'] == state) & (df_enrolment['district'] == wrong)
        df_enrolment.loc[mask, 'district'] = right

# 3. Nuke double spaces and standardize
df_enrolment['district'] = df_enrolment['district'].str.replace(r'\s+', ' ', regex=True).str.strip()

# 4. Save the enrollment file (overwriting the previous version)
df_enrolment.to_csv('Cleaned Datasets/cleaned_enrollment.csv', index=False)

print("Final cleanup applied to Enrollment data.")
print("File 'cleaned_enrollment.csv' has been updated and saved.")

In [ ]:
# Calculate total for display only and group by state/district
enrolment_summary = df_enrolment.groupby(['state', 'district'])[['age_0_5', 'age_5_17', 'age_18_greater']].sum().sum(axis=1)

print("--- ENROLLMENT DISTRICT AUDIT ---")
print(enrolment_summary.sort_values(ascending=False).to_string())

We perform a final audit to confirm that all targeted "survivor" typos have been successfully purged from the enrollment dataset.

<b>Targeted Search:</b> Queried the dataset specifically for the old naming variants (e.g., Bijapur(Kar), Rangareddi) to ensure no records were missed.

<b>Integrity Confirmation:</b> This step verifies that the surgical mapping was applied correctly across the entire DataFrame before moving to the next dataset.

In [ ]:
check_list = ['Kaimur (Bhabua)', 'Bijapur(Kar)', 'K.V. Rangareddy', 'Rangareddi']
remaining = df_enrolment[df_enrolment['district'].isin(check_list)]

if remaining.empty:
    print("Verification: All target typos are gone from Enrollment.")
else:
    print("Still seeing these:")
    print(remaining[['state', 'district']].value_counts())

In [ ]:
# Surgical cleanup for these specific leftovers in df_enrolment
enrolment_mopping = {
    'Telangana': {
        'Yadadri.': 'Yadadri Bhuvanagiri' # Standard name for Yadadri
    },
    'Punjab': {
        'Sas Nagar (Mohali)': 'Sas Nagar' # Cleaning up the parenthesis
    },
    'Maharashtra': {
        'Raigarh(Mh)': 'Raigarh'
    },
    'Jammu And Kashmir': {
        'Leh (Ladakh)': 'Leh'
    }
}

for state, mapping in enrolment_mopping.items():
    for wrong, right in mapping.items():
        mask = (df_enrolment['state'] == state) & (df_enrolment['district'] == wrong)
        df_enrolment.loc[mask, 'district'] = right

df_enrolment['district'] = df_enrolment['district'].str.replace(r'\.', '', regex=True).str.strip()

print("Targeted fixes applied to Enrollment data.")

In [ ]:
audit_states = ['Telangana', 'Punjab', 'Maharashtra', 'Jammu And Kashmir']
enrolment_summary = df_enrolment[df_enrolment['state'].isin(audit_states)].groupby(['state', 'district'])[['age_0_5', 'age_5_17', 'age_18_greater']].sum().sum(axis=1)

print(enrolment_summary.sort_values(ascending=False).to_string())

In [ ]:
df_enrolment.to_csv('Cleaned Datasets/cleaned_enrollment.csv', index=False)
print("File 'cleaned_enrollment.csv' saved successfully.")

In [ ]:
# Create total column for Enrollment
df_enrolment['total_enrolment'] = df_enrolment['age_0_5'] + df_enrolment['age_5_17'] + df_enrolment['age_18_greater']

# Group and print
enrolment_summary = df_enrolment.groupby(['state', 'district'])['total_enrolment'].sum()

print("--- FULL ENROLLMENT LIST BY DISTRICT ---")
print("-" * 60)
print(enrolment_summary.to_string())
print("-" * 60)
print(f"Total Unique Districts in Enrollment: {len(enrolment_summary)}")

In [ ]:
# Final consolidation for the Enrollment survivors
enrolment_final_mopping = {
    'Andhra Pradesh': {
        'Ananthapur': 'Anantapur',
        'Cuddapah': 'Y S R',
        'Spsr Nellore': 'Nellore',
        'Sri Potti Sriramulu Nellore': 'Nellore'
    },
    'Bihar': {
        'Bhabua': 'Kaimur',
        'Monghyr': 'Munger',
        'Pashchim Champaran': 'West Champaran',
        'Purbi Champaran': 'East Champaran'
    },
    'Haryana': {
        'Gurugram': 'Gurgaon',
        'Nuh': 'Mewat'
    },
    'Karnataka': {
        'Bangalore': 'Bengaluru Urban',
        'Belgaum': 'Belagavi',
        'Bellary': 'Ballari',
        'Bengaluru': 'Bengaluru Urban',
        'Bengaluru South': 'Bengaluru Urban',
        'Bijapur': 'Vijayapura',
        'Gulbarga': 'Kalaburagi',
        'Mysore': 'Mysuru'
    },
    'Punjab': {
        'Firozpur': 'Ferozepur',
        'SAS Nagar': 'Sas Nagar (Mohali)',
        'Sas Nagar': 'Sas Nagar (Mohali)',
        'Sri Muktsar Sahib': 'Muktsar'
    },
    'Uttar Pradesh': {
        'Allahabad': 'Prayagraj',
        'Faizabad': 'Ayodhya',
        'Bara Banki': 'Barabanki',
        'Rae Bareli': 'Raebareli',
        'Sant Ravidas Nagar': 'Bhadohi',
        'Siddharth Nagar': 'Siddharthnagar',
        'Shravasti': 'Shrawasti',
        'Jyotiba Phule Nagar': 'Amroha'
    },
    'West Bengal': {
        'Burdwan': 'Bardhaman',
        'Haora': 'Howrah',
        'Hugli': 'Hooghly',
        'Koch Bihar': 'Cooch Behar',
        'Medinipur': 'Paschim Medinipur',
        'Medinipur West': 'Paschim Medinipur',
        'West Medinipur': 'Paschim Medinipur',
        'West Midnapore': 'Paschim Medinipur',
        'Dinajpur Dakshin': 'Dakshin Dinajpur',
        'South Dinajpur': 'Dakshin Dinajpur',
        'Dinajpur Uttar': 'Uttar Dinajpur',
        'North Dinajpur': 'Uttar Dinajpur'
    }
}

# Apply mapping
for state, mapping in enrolment_final_mopping.items():
    for wrong, right in mapping.items():
        df_enrolment.loc[(df_enrolment['state'] == state) & (df_enrolment['district'] == wrong), 'district'] = right

# Special Fix for Sikkim Districts to match Biometric
sikkim_map = {'East': 'East Sikkim', 'West': 'West Sikkim', 'North': 'North Sikkim', 'South': 'South Sikkim'}
for wrong, right in sikkim_map.items():
    df_enrolment.loc[(df_enrolment['state'] == 'Sikkim') & (df_enrolment['district'] == wrong), 'district'] = right

# Clean up whitespace
df_enrolment['district'] = df_enrolment['district'].str.replace(r'\s+', ' ', regex=True).str.strip()

print("Enrollment survivors unified.")

<h3><b>Final Dataset Finalization</b></h3>

After multiple rounds of auditing, fuzzy matching, and surgical error correction, the Enrollment dataset is now fully standardized.

<b>Integrity Guarantee:</b> This final export follows several validation loops, ensuring that "ghost" states and district-level typos are eliminated.

<b>Persistent Storage:</b> The high-integrity DataFrame is saved to CSV, establishing a verified "source of truth" ready for the final master merge.

In [ ]:
# Save the final cleaned enrollment data
df_enrolment.to_csv('Cleaned Datasets/cleaned_enrollment.csv', index=False)
